<a href="https://colab.research.google.com/github/rebeccabynoe/pfizer-document-intelligence-chatbot/blob/main/Final_Project_Ashlee_Gill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install Required Libraries

!pip install -q \
gradio \
pymupdf \
sentence-transformers \
faiss-cpu \
transformers \
accelerate \
pytesseract \
pillow \
rank-bm25 \
pandas \
numpy

!apt-get update -qq
!apt-get install -y tesseract-ocr > /dev/null

print("Required libraries installed successfully.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Required libraries installed successfully.


In [ ]:
# Step 2: Import Libraries and Configure the System

import os
import re
import time
import json
import math
import tempfile
import warnings

import numpy as np
import pandas as pd

import fitz
import faiss
import torch
import gradio as gr
import pytesseract

from PIL import Image, ImageEnhance, ImageFilter

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from rank_bm25 import BM25Okapi

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

warnings.filterwarnings("ignore")


# -------------------------------------------------
# SYSTEM CONFIGURATION
# -------------------------------------------------

APP_NAME = "DocuMind AI"

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

RERANKER_MODEL_NAME = (
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

LLM_MODEL_NAME = "google/flan-t5-base"


# Chunking settings
CHUNK_SIZE = 180
CHUNK_OVERLAP = 40


# Retrieval settings
DEFAULT_TOP_K = 4
INITIAL_CANDIDATES = 12


# OCR settings
OCR_TEXT_THRESHOLD = 50
OCR_SCALE = 2.5


# Select GPU if available
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DocuMind AI configuration ready.")
print("Device:", DEVICE)

DocuMind AI configuration ready.
Device: cpu


In [ ]:
# Step 3: Load the Embedding Model

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE
)

print("Embedding model loaded successfully.")
print("Model:", EMBEDDING_MODEL_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.
Model: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
# Step 4: Load the CrossEncoder Reranker

reranker = CrossEncoder(
    RERANKER_MODEL_NAME,
    device=DEVICE
)

print("Reranker loaded successfully.")
print("Model:", RERANKER_MODEL_NAME)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded successfully.
Model: cross-encoder/ms-marco-MiniLM-L-6-v2


In [ ]:
# Step 5: Load the Open-Source Language Model

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME
)

llm_model = AutoModelForSeq2SeqLM.from_pretrained(
    LLM_MODEL_NAME
)

llm_model = llm_model.to(DEVICE)

llm_model.eval()


def generate_answer(
    prompt,
    max_new_tokens=250
):
    """
    Generate a grounded answer using the
    open-source FLAN-T5 model.
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=2,
            early_stopping=True
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer.strip()


print("Language model loaded successfully.")
print("Model:", LLM_MODEL_NAME)
print("Using device:", DEVICE)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Language model loaded successfully.
Model: google/flan-t5-base
Using device: cpu


In [ ]:
# Step 6: Initialize System State

document_chunks = []
document_metadata = []

faiss_index = None
bm25_index = None
bm25_tokenized_corpus = []

processing_stats = {
    "files": 0,
    "pages": 0,
    "chunks": 0,
    "ocr_pages": 0,
    "processing_time": 0.0
}

print("System state initialized successfully.")

System state initialized successfully.


In [ ]:
# Step 7: Preprocess Images for OCR

def preprocess_for_ocr(image):
    """
    Improve scanned page quality before OCR.
    """

    # Convert to grayscale
    image = image.convert("L")

    # Improve contrast
    enhancer = ImageEnhance.Contrast(image)
    image = enhancer.enhance(2.0)

    # Light sharpening
    image = image.filter(
        ImageFilter.SHARPEN
    )

    # Convert to black and white
    image = image.point(
        lambda x: 0 if x < 170 else 255,
        "1"
    )

    return image


print("OCR preprocessing function ready.")

OCR preprocessing function ready.


In [ ]:
# Step 8: Extract Text from Digital and Scanned PDFs

def extract_pdf_text(file_path):
    """
    Extract text from each PDF page.

    Normal digital extraction is attempted first.
    OCR is automatically used when very little
    readable text is found.
    """

    pdf = fitz.open(file_path)

    extracted_pages = []

    ocr_pages = 0
    digital_pages = 0

    for page_number, page in enumerate(pdf):

        # -----------------------------------------
        # FIRST: TRY DIGITAL TEXT EXTRACTION
        # -----------------------------------------

        text = page.get_text("text").strip()

        used_ocr = False

        # -----------------------------------------
        # OCR FALLBACK
        # -----------------------------------------

        if len(text) < OCR_TEXT_THRESHOLD:

            pix = page.get_pixmap(
                matrix=fitz.Matrix(
                    OCR_SCALE,
                    OCR_SCALE
                ),
                alpha=False
            )

            image = Image.frombytes(
                "RGB",
                [pix.width, pix.height],
                pix.samples
            )

            # Improve image before OCR
            processed_image = preprocess_for_ocr(
                image
            )

            text = pytesseract.image_to_string(
                processed_image,
                config="--psm 6"
            ).strip()

            used_ocr = True
            ocr_pages += 1

        else:
            digital_pages += 1

        # -----------------------------------------
        # STORE PAGE
        # -----------------------------------------

        if text:

            extracted_pages.append({
                "page": page_number + 1,
                "text": text,
                "ocr_used": used_ocr,
                "character_count": len(text)
            })

    total_pdf_pages = len(pdf)

    pdf.close()

    extraction_info = {
        "total_pages": total_pdf_pages,
        "pages_with_text": len(extracted_pages),
        "digital_pages": digital_pages,
        "ocr_pages": ocr_pages
    }

    return extracted_pages, extraction_info

In [ ]:
# Step 9: Automatically Classify Pharmaceutical Documents

def classify_document(pages):
    """
    Classify an uploaded document using
    pharmaceutical document keywords.
    """

    if not pages:
        return "General Document"

    # Combine a sample of text from the first pages
    sample_text = " ".join(
        page["text"]
        for page in pages[:3]
    ).lower()

    # -----------------------------------------
    # DOCUMENT TYPE KEYWORDS
    # -----------------------------------------

    document_rules = {

        "Certificate of Quality": [
            "certificate of quality",
            "certificate of analysis",
            "lot number",
            "batch number",
            "test result",
            "specification result",
            "manufacture date",
            "expiration date"
        ],

        "Packaging Specification": [
            "packaging specification",
            "packaging material",
            "container",
            "closure system",
            "part number",
            "pack configuration",
            "packaging component"
        ],

        "BSE/TSE Declaration": [
            "bse",
            "tse",
            "bovine spongiform",
            "transmissible spongiform",
            "animal origin",
            "animal-derived"
        ],

        "Cover Letter": [
            "dear sir",
            "dear madam",
            "to whom it may concern",
            "sincerely",
            "subject:",
            "regarding:"
        ],

        "Compliance Document": [
            "compliance",
            "regulatory",
            "declaration",
            "certification",
            "conforms",
            "complies with"
        ],

        "Technical Report": [
            "technical report",
            "method",
            "procedure",
            "operating condition",
            "technical data",
            "performance"
        ]
    }

    scores = {}

    # -----------------------------------------
    # SCORE EACH DOCUMENT TYPE
    # -----------------------------------------

    for doc_type, keywords in document_rules.items():

        score = sum(
            1
            for keyword in keywords
            if keyword in sample_text
        )

        scores[doc_type] = score

    # Find best match
    best_type = max(
        scores,
        key=scores.get
    )

    # If nothing matched, use general
    if scores[best_type] == 0:
        return "General Document"

    return best_type

In [ ]:
# Step 10: Clean and Normalize Extracted Text

def clean_text(text):
    """
    Clean PDF/OCR text before chunking.
    """

    if not text:
        return ""

    # Remove null characters
    text = text.replace("\x00", " ")

    # Replace repeated line breaks
    text = re.sub(
        r"\n+",
        " ",
        text
    )

    # Replace repeated tabs
    text = re.sub(
        r"\t+",
        " ",
        text
    )

    # Normalize repeated whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove spaces before punctuation
    text = re.sub(
        r"\s+([.,!?;:%])",
        r"\1",
        text
    )

    # Normalize spaces after punctuation
    text = re.sub(
        r"([.,!?;:])(?=[A-Za-z])",
        r"\1 ",
        text
    )

    return text.strip()


print("Text cleaning function ready.")

Text cleaning function ready.


In [ ]:
# Step 11: Intelligent Chunking and Rich Metadata

def create_chunks(
    pages,
    source_name,
    document_type,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
):
    """
    Split page text into overlapping chunks
    and attach detailed metadata.
    """

    chunks = []

    for page_data in pages:

        cleaned_text = clean_text(
            page_data["text"]
        )

        words = cleaned_text.split()

        if not words:
            continue

        start = 0
        chunk_number = 1

        while start < len(words):

            end = start + chunk_size

            chunk_words = words[start:end]

            chunk_text = " ".join(
                chunk_words
            ).strip()

            if chunk_text:

                chunk_id = (
                    f"{source_name}"
                    f"_p{page_data['page']}"
                    f"_c{chunk_number}"
                )

                chunk = {
                    "text": chunk_text,
                    "source_file": source_name,
                    "document_type": document_type,
                    "page_start": page_data["page"],
                    "page_end": page_data["page"],
                    "chunk_id": chunk_id,
                    "ocr_used": page_data["ocr_used"],
                    "word_count": len(chunk_words),
                    "character_count": len(chunk_text)
                }

                chunks.append(chunk)

            # Keep overlap between chunks
            start += chunk_size - overlap
            chunk_number += 1

    return chunks


print("Chunking function ready.")

Chunking function ready.


In [ ]:
# Step 12: Build Semantic and Keyword Indexes


# =========================================================
# BM25 TOKENIZER
# =========================================================

def tokenize_for_bm25(text):
    """
    Prepare text for BM25 keyword retrieval.

    Keeps useful pharmaceutical values such as
    temperatures, percentages, product codes,
    numbers, and technical terms.
    """

    if not text:
        return []

    text = text.lower()

    tokens = re.findall(
        r"\b[a-z0-9°%./+-]+\b",
        text
    )

    return tokens


# =========================================================
# BUILD FAISS + BM25 INDEXES
# =========================================================

def build_indexes(chunks):
    """
    Build two indexes:

    1. FAISS semantic vector index
    2. BM25 keyword search index
    """

    global document_chunks
    global document_metadata
    global faiss_index
    global bm25_index
    global bm25_tokenized_corpus

    # -----------------------------------------------------
    # CHECK INPUT
    # -----------------------------------------------------

    if not chunks:
        raise ValueError(
            "No document chunks were provided."
        )

    # -----------------------------------------------------
    # STORE DOCUMENT DATA
    # -----------------------------------------------------

    document_chunks = [
        chunk["text"]
        for chunk in chunks
    ]

    document_metadata = chunks


    # -----------------------------------------------------
    # CREATE SEMANTIC EMBEDDINGS
    # -----------------------------------------------------

    print("Generating semantic embeddings...")

    embeddings = embedding_model.encode(
        document_chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")


    # -----------------------------------------------------
    # BUILD FAISS INDEX
    # -----------------------------------------------------

    dimension = embeddings.shape[1]

    faiss_index = faiss.IndexFlatIP(
        dimension
    )

    faiss_index.add(
        embeddings
    )

    print(
        "FAISS semantic index created."
    )


    # -----------------------------------------------------
    # TOKENIZE DOCUMENTS FOR BM25
    # -----------------------------------------------------

    print(
        "Preparing documents for "
        "keyword retrieval..."
    )

    bm25_tokenized_corpus = [
        tokenize_for_bm25(text)
        for text in document_chunks
    ]


    # -----------------------------------------------------
    # BUILD BM25 INDEX
    # -----------------------------------------------------

    bm25_index = BM25Okapi(
        bm25_tokenized_corpus
    )

    print(
        "BM25 keyword index created."
    )


    # -----------------------------------------------------
    # VERIFY INDEXES
    # -----------------------------------------------------

    print()
    print(
        "Indexes built successfully."
    )

    print(
        "Chunks stored:",
        len(document_chunks)
    )

    print(
        "Vectors stored in FAISS:",
        faiss_index.ntotal
    )

    print(
        "Documents stored in BM25:",
        len(
            bm25_tokenized_corpus
        )
    )

In [ ]:
# Step 13: Hybrid Retrieval

def hybrid_retrieve(
    query,
    candidate_k=INITIAL_CANDIDATES,
    document_type_filter="All Documents"
):
    """
    Retrieve candidate chunks using a combination
    of semantic FAISS search and BM25 keyword search.
    """

    if (
        faiss_index is None
        or bm25_index is None
        or not document_chunks
    ):
        return []

    candidate_k = min(
        candidate_k,
        len(document_chunks)
    )

    # -----------------------------------------
    # SEMANTIC SEARCH
    # -----------------------------------------

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    semantic_scores, semantic_indices = (
        faiss_index.search(
            query_embedding,
            candidate_k
        )
    )

    # -----------------------------------------
    # BM25 SEARCH
    # -----------------------------------------

    query_tokens = tokenize_for_bm25(
        query
    )

    bm25_scores = bm25_index.get_scores(
        query_tokens
    )

    bm25_ranked_indices = np.argsort(
        bm25_scores
    )[::-1][:candidate_k]

    # -----------------------------------------
    # COMBINE CANDIDATES
    # -----------------------------------------

    combined = {}

    # Add semantic candidates
    for rank, (
        score,
        index
    ) in enumerate(
        zip(
            semantic_scores[0],
            semantic_indices[0]
        )
    ):

        if index < 0:
            continue

        combined[index] = {
            "semantic_score": float(score),
            "bm25_score": 0.0,
            "semantic_rank": rank + 1,
            "bm25_rank": None
        }

    # Add BM25 candidates
    for rank, index in enumerate(
        bm25_ranked_indices
    ):

        if index not in combined:

            combined[index] = {
                "semantic_score": 0.0,
                "bm25_score": float(
                    bm25_scores[index]
                ),
                "semantic_rank": None,
                "bm25_rank": rank + 1
            }

        else:

            combined[index]["bm25_score"] = (
                float(
                    bm25_scores[index]
                )
            )

            combined[index]["bm25_rank"] = (
                rank + 1
            )

    # -----------------------------------------
    # NORMALIZE BM25 SCORES
    # -----------------------------------------

    max_bm25 = max(
        [
            item["bm25_score"]
            for item in combined.values()
        ]
        + [1.0]
    )

    results = []

    for index, scores in combined.items():

        metadata = document_metadata[
            index
        ]

        # Optional metadata filter
        if (
            document_type_filter
            and
            document_type_filter
            != "All Documents"
            and
            metadata["document_type"]
            != document_type_filter
        ):
            continue

        normalized_bm25 = (
            scores["bm25_score"]
            / max_bm25
        )

        # Hybrid weighted score
        hybrid_score = (
            0.70
            * scores["semantic_score"]
            +
            0.30
            * normalized_bm25
        )

        results.append({
            **metadata,
            "semantic_score":
                scores["semantic_score"],
            "bm25_score":
                scores["bm25_score"],
            "hybrid_score":
                float(hybrid_score)
        })

    # Highest hybrid score first
    results.sort(
        key=lambda x:
            x["hybrid_score"],
        reverse=True
    )

    return results[:candidate_k]

In [ ]:
# Step 14: CrossEncoder Reranking

def rerank_chunks(
    query,
    candidates,
    top_k=DEFAULT_TOP_K
):
    """
    Rerank retrieved candidate chunks using
    a CrossEncoder model.
    """

    if not candidates:
        return []

    # Build question + chunk pairs
    pairs = [
        (
            query,
            candidate["text"]
        )
        for candidate in candidates
    ]

    # CrossEncoder scores each pair
    rerank_scores = reranker.predict(
        pairs
    )

    reranked = []

    for candidate, score in zip(
        candidates,
        rerank_scores
    ):

        result = candidate.copy()

        result["rerank_score"] = float(
            score
        )

        reranked.append(
            result
        )

    # Sort by reranker score
    reranked.sort(
        key=lambda x:
            x["rerank_score"],
        reverse=True
    )

    return reranked[:top_k]

In [ ]:
# Step 15: Smart Query Routing

def route_query(query):
    """
    Determine the best retrieval strategy
    for the user's question.
    """

    query_lower = query.lower().strip()

    # -----------------------------------------
    # PAGE LOOKUP
    # -----------------------------------------

    if re.search(
        r"\bpage\s+\d+\b",
        query_lower
    ):
        return "page_lookup"

    # -----------------------------------------
    # COMPARISON
    # -----------------------------------------

    if any(
        phrase in query_lower
        for phrase in [
            "compare",
            "difference between",
            "differences between",
            "versus",
            " vs ",
            "similarities",
            "how are they different"
        ]
    ):
        return "comparison"

    # -----------------------------------------
    # SUMMARY
    # -----------------------------------------

    if any(
        phrase in query_lower
        for phrase in [
            "summarize",
            "summary",
            "overview",
            "main points",
            "key points",
            "what is this document about",
            "what is the document about",
            "what is this about"
        ]
    ):
        return "summary"

    # -----------------------------------------
    # EVIDENCE / SOURCE LOOKUP
    # -----------------------------------------

    if any(
        phrase in query_lower
        for phrase in [
            "evidence",
            "source",
            "where does it say",
            "which page",
            "cite",
            "citation",
            "show me where",
            "where is"
        ]
    ):
        return "evidence"

    # -----------------------------------------
    # DEFAULT
    # -----------------------------------------

    return "general"


print("Smart query router ready.")

Smart query router ready.


In [ ]:
# Step 16: Route-Aware Retrieval

def retrieve_for_route(
    query,
    route,
    top_k=DEFAULT_TOP_K,
    document_type_filter="All Documents"
):
    """
    Retrieve evidence differently depending
    on the type of user question.
    """

    # -----------------------------------------
    # PAGE LOOKUP
    # -----------------------------------------

    if route == "page_lookup":

        page_match = re.search(
            r"\bpage\s+(\d+)\b",
            query.lower()
        )

        if page_match:

            page_number = int(
                page_match.group(1)
            )

            page_results = []

            for chunk in document_metadata:

                if (
                    chunk["page_start"]
                    == page_number
                ):

                    if (
                        document_type_filter
                        != "All Documents"
                        and
                        chunk["document_type"]
                        != document_type_filter
                    ):
                        continue

                    result = chunk.copy()

                    result[
                        "hybrid_score"
                    ] = 1.0

                    result[
                        "rerank_score"
                    ] = 1.0

                    page_results.append(
                        result
                    )

            return page_results[
                :top_k
            ]


    # -----------------------------------------
    # SUMMARY
    # -----------------------------------------

    if route == "summary":

        filtered_chunks = []

        for chunk in document_metadata:

            if (
                document_type_filter
                != "All Documents"
                and
                chunk["document_type"]
                != document_type_filter
            ):
                continue

            filtered_chunks.append(
                chunk
            )

        if not filtered_chunks:
            return []

        # Select chunks from different
        # parts of the document instead
        # of using only similarity search

        total = len(
            filtered_chunks
        )

        positions = np.linspace(
            0,
            total - 1,
            min(top_k, total),
            dtype=int
        )

        summary_results = []

        for position in positions:

            result = filtered_chunks[
                position
            ].copy()

            result[
                "hybrid_score"
            ] = 1.0

            result[
                "rerank_score"
            ] = 1.0

            summary_results.append(
                result
            )

        return summary_results


    # -----------------------------------------
    # GENERAL / EVIDENCE / COMPARISON
    # -----------------------------------------

    candidates = hybrid_retrieve(
        query=query,
        candidate_k=INITIAL_CANDIDATES,
        document_type_filter=
            document_type_filter
    )

    reranked = rerank_chunks(
        query=query,
        candidates=candidates,
        top_k=top_k
    )

    return reranked

In [ ]:
# Step 17: Build Focused Context and Grounded Prompt

def shorten_evidence(text, max_chars=900):
    """
    Keep evidence short enough that the LLM
    can focus on the most useful information.
    """

    text = clean_text(text)

    if len(text) <= max_chars:
        return text

    return text[:max_chars] + "..."


def build_context(retrieved_chunks):
    """
    Build compact evidence context for the LLM.
    """

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks[:4], 1):

        source_name = chunk["source_file"]
        page_start = chunk["page_start"]
        page_end = chunk["page_end"]

        if page_start == page_end:
            page_label = str(page_start)
        else:
            page_label = f"{page_start}-{page_end}"

        evidence_text = shorten_evidence(
            chunk["text"]
        )

        context_parts.append(
            f"""
EVIDENCE {i}
Source: {source_name}
Page: {page_label}
Content: {evidence_text}
"""
        )

    return "\n".join(context_parts)


def build_rag_prompt(
    query,
    retrieved_chunks,
    route
):
    """
    Build a short, question-focused RAG prompt.
    """

    context = build_context(
        retrieved_chunks
    )

    if route == "summary":

        instruction = """
Summarize the main information across the evidence.
Do not copy headings or repeat raw document text.
"""

    elif route == "evidence":

        instruction = """
Answer the question directly, then identify the
strongest source and page supporting the answer.
"""

    elif route == "comparison":

        instruction = """
Compare only the information supported by the
evidence. Clearly state the differences.
"""

    elif route == "page_lookup":

        instruction = """
Explain the important information found on the
requested page.
"""

    else:

        instruction = """
Give a direct factual answer to the question.

If the question asks for a number, temperature,
date, range, product name, lot number, or other
specific value, state that value first.

Do not begin by copying document headings,
email subjects, greetings, or titles.
"""

    prompt = f"""
You are DocuMind AI.

Answer the user's question using ONLY the evidence.

IMPORTANT RULES:
- Answer the exact question asked.
- State the key fact first.
- Do not copy irrelevant headings or greetings.
- Do not use outside knowledge.
- Do not invent information.
- Cite the source file and page.
- If the evidence does not contain the answer, say:
  "I could not find enough evidence in the uploaded
  documents to answer that reliably."

INSTRUCTIONS:
{instruction}

EVIDENCE:
{context}

QUESTION:
{query}

DIRECT ANSWER:
"""

    return prompt


print("Improved focused prompt builder ready.")

Improved focused prompt builder ready.


In [ ]:
# Step 18: Generate Final RAG Response

def calculate_retrieval_confidence(
    retrieved_chunks
):
    """
    Convert retrieval scores into a simple
    user-facing confidence indicator.
    """

    if not retrieved_chunks:
        return 0.0, "LOW"

    available_scores = []

    for chunk in retrieved_chunks:

        # Prefer normalized hybrid score
        # because reranker scores are not
        # naturally bounded between 0 and 1

        score = chunk.get(
            "hybrid_score",
            0.0
        )

        available_scores.append(
            max(
                0.0,
                min(1.0, score)
            )
        )

    confidence = float(
        np.mean(
            available_scores
        )
    )

    if confidence >= 0.65:
        label = "HIGH"

    elif confidence >= 0.35:
        label = "MODERATE"

    else:
        label = "LOW"

    return confidence, label


def rag_answer(
    query,
    top_k=DEFAULT_TOP_K,
    document_type_filter=
        "All Documents",
    auto_route=True
):
    """
    Complete RAG pipeline:
    Route -> Retrieve -> Rerank ->
    Context -> Prompt -> LLM -> Response
    """

    if not document_metadata:

        return {
            "answer":
                "Please upload and process "
                "documents first.",

            "sources": [],
            "evidence": [],
            "retrieval_confidence": 0.0,
            "confidence_label": "LOW",
            "chunks_used": 0,
            "route": "none",
            "retrieval_time": 0.0,
            "generation_time": 0.0,
            "total_time": 0.0
        }

    total_start = time.time()

    # -----------------------------------------
    # ROUTING
    # -----------------------------------------

    if auto_route:
        route = route_query(
            query
        )
    else:
        route = "general"

    # -----------------------------------------
    # RETRIEVAL
    # -----------------------------------------

    retrieval_start = time.time()

    retrieved = retrieve_for_route(
        query=query,
        route=route,
        top_k=top_k,
        document_type_filter=
            document_type_filter
    )

    retrieval_time = (
        time.time()
        - retrieval_start
    )

    if not retrieved:

        return {
            "answer":
                "I could not find enough "
                "evidence in the uploaded "
                "documents to answer that "
                "reliably.",

            "sources": [],
            "evidence": [],
            "retrieval_confidence": 0.0,
            "confidence_label": "LOW",
            "chunks_used": 0,
            "route": route,
            "retrieval_time":
                round(
                    retrieval_time,
                    3
                ),
            "generation_time": 0.0,
            "total_time":
                round(
                    time.time()
                    - total_start,
                    3
                )
        }

    # -----------------------------------------
    # PROMPT
    # -----------------------------------------

    prompt = build_rag_prompt(
        query=query,
        retrieved_chunks=retrieved,
        route=route
    )

    # -----------------------------------------
    # GENERATION
    # -----------------------------------------

    generation_start = (
        time.time()
    )

    answer = generate_answer(
        prompt,
        max_new_tokens=250
    )

    generation_time = (
        time.time()
        - generation_start
    )

    # -----------------------------------------
    # CONFIDENCE
    # -----------------------------------------

    confidence, confidence_label = (
        calculate_retrieval_confidence(
            retrieved
        )
    )

    # -----------------------------------------
    # SOURCES
    # -----------------------------------------

    sources = []

    for chunk in retrieved:

        source = (
            f"{chunk['source_file']} "
            f"— Page "
            f"{chunk['page_start']}"
        )

        if source not in sources:
            sources.append(
                source
            )

    total_time = (
        time.time()
        - total_start
    )

    return {
        "answer": answer,
        "sources": sources,
        "evidence": retrieved,
        "retrieval_confidence":
            confidence,
        "confidence_label":
            confidence_label,
        "chunks_used":
            len(retrieved),
        "route": route,
        "retrieval_time":
            round(
                retrieval_time,
                3
            ),
        "generation_time":
            round(
                generation_time,
                3
            ),
        "total_time":
            round(
                total_time,
                3
            )
    }


print(
    "Final RAG response "
    "function ready."
)

Final RAG response function ready.


In [ ]:
# Step 19: Process Documents End-to-End

def process_documents(
    files,
    document_type_override=
        "Automatic Detection"
):
    """
    Process all uploaded PDFs and build
    the complete retrieval indexes.
    """

    global processing_stats

    if not files:

        return {
            "status":
                "Please upload at least "
                "one PDF.",

            "files": 0,
            "pages": 0,
            "chunks": 0,
            "ocr_pages": 0,
            "processing_time": 0.0,
            "documents": []
        }

    start_time = time.time()

    all_chunks = []
    document_reports = []

    total_pages = 0
    total_ocr_pages = 0

    for file_path in files:

        source_name = os.path.basename(
            file_path
        )

        # -------------------------------------
        # EXTRACT
        # -------------------------------------

        pages, extraction_info = (
            extract_pdf_text(
                file_path
            )
        )

        # -------------------------------------
        # CLASSIFY
        # -------------------------------------

        detected_type = (
            classify_document(
                pages
            )
        )

        if (
            document_type_override
            != "Automatic Detection"
        ):

            final_type = (
                document_type_override
            )

        else:

            final_type = (
                detected_type
            )

        # -------------------------------------
        # CHUNK
        # -------------------------------------

        chunks = create_chunks(
            pages=pages,
            source_name=
                source_name,
            document_type=
                final_type
        )

        all_chunks.extend(
            chunks
        )

        total_pages += (
            extraction_info[
                "total_pages"
            ]
        )

        total_ocr_pages += (
            extraction_info[
                "ocr_pages"
            ]
        )

        document_reports.append({
            "source_file":
                source_name,
            "detected_type":
                detected_type,
            "final_type":
                final_type,
            "pages":
                extraction_info[
                    "total_pages"
                ],
            "digital_pages":
                extraction_info[
                    "digital_pages"
                ],
            "ocr_pages":
                extraction_info[
                    "ocr_pages"
                ],
            "chunks":
                len(chunks)
        })

    # -----------------------------------------
    # INDEX EVERYTHING
    # -----------------------------------------

    if not all_chunks:

        return {
            "status":
                "No readable text could "
                "be extracted from the "
                "uploaded documents.",

            "files":
                len(files),
            "pages":
                total_pages,
            "chunks": 0,
            "ocr_pages":
                total_ocr_pages,
            "processing_time":
                round(
                    time.time()
                    - start_time,
                    2
                ),
            "documents":
                document_reports
        }

    build_indexes(
        all_chunks
    )

    elapsed = (
        time.time()
        - start_time
    )

    processing_stats = {
        "files": len(files),
        "pages": total_pages,
        "chunks":
            len(all_chunks),
        "ocr_pages":
            total_ocr_pages,
        "processing_time":
            round(
                elapsed,
                2
            )
    }

    return {
        "status":
            "Documents processed "
            "successfully.",
        **processing_stats,
        "documents":
            document_reports
    }


print(
    "End-to-end document "
    "processor ready."
)

End-to-end document processor ready.


In [ ]:
# Step 20: RAG Evaluation Engine

def normalize_answer_text(
    text
):
    """
    Normalize text for simple answer
    comparison.
    """

    if text is None:
        return ""

    text = str(
        text
    ).lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    text = re.sub(
        r"[^\w\s°%+.-]",
        "",
        text
    )

    return text.strip()


def evaluate_single_query(
    test_case,
    top_k=DEFAULT_TOP_K
):
    """
    Evaluate one ground-truth question.
    """

    question = test_case[
        "question"
    ]

    expected_answer = test_case.get(
        "expected_answer",
        ""
    )

    expected_source = test_case.get(
        "expected_source",
        ""
    )

    expected_page = test_case.get(
        "expected_page",
        None
    )

    result = rag_answer(
        query=question,
        top_k=top_k,
        document_type_filter=
            "All Documents",
        auto_route=True
    )

    # -----------------------------------------
    # ANSWER MATCH
    # -----------------------------------------

    generated_normalized = (
        normalize_answer_text(
            result["answer"]
        )
    )

    expected_normalized = (
        normalize_answer_text(
            expected_answer
        )
    )

    answer_correct = False

    if expected_normalized:

        answer_correct = (
            expected_normalized
            in generated_normalized
        )

    # -----------------------------------------
    # RETRIEVAL RELEVANCE
    # -----------------------------------------

    relevant_ranks = []

    relevant_count = 0

    for rank, chunk in enumerate(
        result["evidence"],
        1
    ):

        source_match = (
            not expected_source
            or
            chunk["source_file"]
            == expected_source
        )

        page_match = (
            expected_page is None
            or
            chunk["page_start"]
            == expected_page
            or
            chunk["page_end"]
            == expected_page
        )

        relevant = (
            source_match
            and page_match
        )

        if relevant:

            relevant_count += 1

            relevant_ranks.append(
                rank
            )

    # Recall@K:
    # Did at least one expected source/page
    # appear in the retrieved top K?

    recall_at_k = (
        1
        if relevant_count > 0
        else 0
    )

    # Precision@K:
    # Fraction of retrieved chunks that
    # match the expected source/page

    precision_at_k = (
        relevant_count
        / len(result["evidence"])
        if result["evidence"]
        else 0.0
    )

    # Reciprocal rank

    if relevant_ranks:

        reciprocal_rank = (
            1
            / min(
                relevant_ranks
            )
        )

        first_relevant_rank = (
            min(
                relevant_ranks
            )
        )

    else:

        reciprocal_rank = 0.0
        first_relevant_rank = None

    # -----------------------------------------
    # CITATION / SOURCE ACCURACY
    # -----------------------------------------

    source_correct = False

    if expected_source:

        source_correct = any(
            expected_source
            in source
            for source
            in result["sources"]
        )

    else:

        source_correct = bool(
            result["sources"]
        )

    # -----------------------------------------
    # RETURN TEST RESULT
    # -----------------------------------------

    return {
        "question":
            question,
        "expected_answer":
            expected_answer,
        "generated_answer":
            result["answer"],
        "answer_correct":
            answer_correct,
        "expected_source":
            expected_source,
        "expected_page":
            expected_page,
        "source_correct":
            source_correct,
        "recall_at_k":
            recall_at_k,
        "precision_at_k":
            precision_at_k,
        "first_relevant_rank":
            first_relevant_rank,
        "reciprocal_rank":
            reciprocal_rank,
        "retrieval_confidence":
            result[
                "retrieval_confidence"
            ],
        "confidence_label":
            result[
                "confidence_label"
            ],
        "chunks_used":
            result[
                "chunks_used"
            ],
        "retrieval_time":
            result[
                "retrieval_time"
            ],
        "generation_time":
            result[
                "generation_time"
            ],
        "total_time":
            result[
                "total_time"
            ]
    }

In [ ]:
def run_evaluation(
    test_set,
    top_k=DEFAULT_TOP_K
):
    """
    Run the entire ground-truth test set
    and calculate final metrics.
    """

    results = []

    for number, test_case in enumerate(
        test_set,
        1
    ):

        print(
            f"Evaluating "
            f"{number}/"
            f"{len(test_set)}..."
        )

        result = (
            evaluate_single_query(
                test_case,
                top_k=top_k
            )
        )

        results.append(
            result
        )

    results_df = pd.DataFrame(
        results
    )

    if results_df.empty:

        return (
            results_df,
            {}
        )

    metrics = {

        "questions_tested":
            len(results_df),

        "answer_accuracy":
            float(
                results_df[
                    "answer_correct"
                ].mean()
            ),

        "citation_accuracy":
            float(
                results_df[
                    "source_correct"
                ].mean()
            ),

        "recall_at_k":
            float(
                results_df[
                    "recall_at_k"
                ].mean()
            ),

        "precision_at_k":
            float(
                results_df[
                    "precision_at_k"
                ].mean()
            ),

        "mrr":
            float(
                results_df[
                    "reciprocal_rank"
                ].mean()
            ),

        "avg_retrieval_time":
            float(
                results_df[
                    "retrieval_time"
                ].mean()
            ),

        "avg_generation_time":
            float(
                results_df[
                    "generation_time"
                ].mean()
            ),

        "avg_total_time":
            float(
                results_df[
                    "total_time"
                ].mean()
            )
    }

    return (
        results_df,
        metrics
    )


print(
    "Evaluation engine ready."
)

Evaluation engine ready.


In [ ]:
# Step 21: Build the Final DocuMind AI Interface

# =========================================================
# IMPORT TRACEBACK FOR BETTER ERROR REPORTING
# =========================================================

import traceback


# =========================================================
# CUSTOM DESIGN
# =========================================================

custom_css = """
body {
    background: #F5F8FF;
}

.gradio-container {
    max-width: 1380px !important;
    margin: auto !important;
    background: #F5F8FF !important;
}

/* Header */
#brand-title {
    font-size: 38px;
    font-weight: 750;
    color: #0F172A;
    margin-bottom: 0px;
}

#brand-subtitle {
    color: #64748B;
    font-size: 16px;
    margin-top: 0px;
    margin-bottom: 18px;
}

#system-ready {
    background: #ECFDF5;
    color: #15803D;
    border: 1px solid #BBF7D0;
    border-radius: 999px;
    padding: 8px 14px;
    font-size: 13px;
    font-weight: 600;
    text-align: center;
}

/* Main cards */
.card {
    background: #FFFFFF;
    border: 1px solid #DBEAFE;
    border-radius: 18px;
    padding: 18px;
    box-shadow: 0 5px 18px rgba(37, 99, 235, 0.06);
}

/* Metric cards */
.metric-card {
    background: #FFFFFF;
    border: 1px solid #DBEAFE;
    border-radius: 14px;
    padding: 14px;
    min-height: 90px;
    box-shadow: 0 4px 12px rgba(37, 99, 235, 0.05);
}

.metric-label {
    color: #64748B;
    font-size: 12px;
    font-weight: 700;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

.metric-value {
    color: #1E3A8A;
    font-size: 28px;
    font-weight: 750;
}

/* Blue button */
.primary-button {
    background: linear-gradient(
        135deg,
        #2563EB,
        #1D4ED8
    ) !important;

    color: white !important;
    border: none !important;
    border-radius: 11px !important;
    font-weight: 650 !important;
}

/* Secondary buttons */
.secondary-button {
    border-radius: 11px !important;
}

/* Status panel */
#processing-status {
    background: #EFF6FF;
    border: 1px solid #BFDBFE;
    border-radius: 14px;
    padding: 14px;
}

/* Confidence card */
#answer-details {
    background: #F8FAFC;
    border: 1px solid #E2E8F0;
    border-radius: 14px;
    padding: 14px;
}

/* Evidence box */
#evidence-output {
    background: #F8FAFC;
    border: 1px solid #DBEAFE;
    border-radius: 14px;
    padding: 14px;
}

/* Evaluation metrics */
#evaluation-summary {
    background: #EFF6FF;
    border: 1px solid #BFDBFE;
    border-radius: 14px;
    padding: 16px;
}

/* Footer */
#footer {
    color: #64748B;
    text-align: center;
    font-size: 13px;
    margin-top: 22px;
    padding-bottom: 20px;
}
"""


# =========================================================
# PROCESSING REPORT FORMATTER
# =========================================================

def format_processing_report(report):
    """
    Convert processing results into
    user-friendly dashboard output.
    """

    if not report:
        return (
            "### Processing Error\nNo processing report was returned.",
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "No document report available."
        )

    if report.get("files", 0) == 0:
        return (
            report.get(
                "status",
                "No documents processed."
            ),
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "No documents processed."
        )

    document_lines = []

    for doc in report.get(
        "documents",
        []
    ):

        document_lines.append(
            f"**{doc['source_file']}**  \n"
            f"Detected Type: {doc['detected_type']}  \n"
            f"Final Type: {doc['final_type']}  \n"
            f"Pages: {doc['pages']}  \n"
            f"Digital Pages: {doc['digital_pages']}  \n"
            f"OCR Pages: {doc['ocr_pages']}  \n"
            f"Chunks: {doc['chunks']}\n"
        )

    document_report = "\n---\n".join(
        document_lines
    )

    status_text = (
        f"### System Ready\n\n"
        f"{report.get('status', 'Documents processed successfully.')}  \n\n"
        f"**Processing time:** "
        f"{report.get('processing_time', 0)} seconds"
    )

    files_html = (
        f"<div class='metric-value'>"
        f"{report.get('files', 0)}"
        f"</div>"
    )

    pages_html = (
        f"<div class='metric-value'>"
        f"{report.get('pages', 0)}"
        f"</div>"
    )

    chunks_html = (
        f"<div class='metric-value'>"
        f"{report.get('chunks', 0)}"
        f"</div>"
    )

    ocr_html = (
        f"<div class='metric-value'>"
        f"{report.get('ocr_pages', 0)}"
        f"</div>"
    )

    return (
        status_text,
        files_html,
        pages_html,
        chunks_html,
        ocr_html,
        document_report
    )


# =========================================================
# UPDATED SAFE DOCUMENT PROCESSING FUNCTION
# =========================================================

def process_documents_ui(
    files,
    document_type_override
):
    """
    Safely process uploaded documents.
    Displays the real Python error if anything fails.
    """

    try:

        # -----------------------------------------
        # CHECK INPUT
        # -----------------------------------------

        if not files:

            return (
                "### Processing Error\n\n"
                "No PDF files were uploaded.",
                "<div class='metric-value'>0</div>",
                "<div class='metric-value'>0</div>",
                "<div class='metric-value'>0</div>",
                "<div class='metric-value'>0</div>",
                "No document report available."
            )

        # Make sure files is always a list
        if not isinstance(
            files,
            list
        ):
            files = [files]

        cleaned_files = []

        # -----------------------------------------
        # HANDLE DIFFERENT GRADIO FILE FORMATS
        # -----------------------------------------

        for file in files:

            if isinstance(
                file,
                str
            ):

                file_path = file

            elif hasattr(
                file,
                "name"
            ):

                file_path = file.name

            elif hasattr(
                file,
                "path"
            ):

                file_path = file.path

            else:

                raise ValueError(
                    "Unsupported uploaded file "
                    f"type: {type(file)}"
                )

            if not os.path.exists(
                file_path
            ):

                raise FileNotFoundError(
                    "Uploaded file could not "
                    f"be found: {file_path}"
                )

            cleaned_files.append(
                file_path
            )

        print(
            "FILES RECEIVED:"
        )

        print(
            cleaned_files
        )

        # -----------------------------------------
        # RUN FULL DOCUMENT PIPELINE
        # -----------------------------------------

        report = process_documents(
            files=cleaned_files,
            document_type_override=
                document_type_override
        )

        print(
            "PROCESSING REPORT:"
        )

        print(
            report
        )

        return format_processing_report(
            report
        )

    except Exception as error:

        # Full traceback in Colab output
        traceback.print_exc()

        error_message = (
            "### Processing Error\n\n"
            f"**{type(error).__name__}:** "
            f"{str(error)}\n\n"
            "The full traceback is also printed "
            "below this Colab cell."
        )

        return (
            error_message,
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            "<div class='metric-value'>0</div>",
            f"**Processing failed:** {str(error)}"
        )


# =========================================================
# CHAT FUNCTION
# =========================================================

def chat_ui(
    message,
    history,
    top_k,
    document_filter,
    auto_route
):

    if not message or not message.strip():

        return (
            history,
            "",
            "No answer details yet.",
            "No evidence retrieved yet."
        )

    history = history or []

    history.append({
        "role": "user",
        "content": message
    })

    result = rag_answer(
        query=message,
        top_k=int(top_k),
        document_type_filter=
            document_filter,
        auto_route=auto_route
    )

    answer = result[
        "answer"
    ]

    # -----------------------------------------
    # ADD SOURCES
    # -----------------------------------------

    if result[
        "sources"
    ]:

        answer += (
            "\n\n### Sources\n"
        )

        for source in result[
            "sources"
        ]:

            answer += (
                f"- {source}\n"
            )

    history.append({
        "role": "assistant",
        "content": answer
    })

    # -----------------------------------------
    # RETRIEVAL DETAILS
    # -----------------------------------------

    confidence_percentage = round(
        result[
            "retrieval_confidence"
        ] * 100,
        1
    )

    confidence_label = result[
        "confidence_label"
    ]

    if confidence_label == "HIGH":
        confidence_icon = "●"

    elif confidence_label == "MODERATE":
        confidence_icon = "◐"

    else:
        confidence_icon = "○"

    details = f"""
### Retrieval Details

**{confidence_icon} Retrieval Confidence:**
{confidence_percentage}% — **{confidence_label}**

**Query Route:**
{result['route'].replace("_", " ").title()}

**Evidence Chunks Used:**
{result['chunks_used']}

**Retrieval Time:**
{result['retrieval_time']} sec

**Generation Time:**
{result['generation_time']} sec

**Total Response Time:**
{result['total_time']} sec

*Retrieval confidence reflects similarity between the
question and retrieved evidence. It is not the probability
that the final answer is correct.*
"""

    # -----------------------------------------
    # EVIDENCE DISPLAY
    # -----------------------------------------

    evidence_markdown = ""

    if not result[
        "evidence"
    ]:

        evidence_markdown = (
            "No evidence was retrieved."
        )

    else:

        for number, chunk in enumerate(
            result["evidence"],
            1
        ):

            hybrid_score = round(
                chunk.get(
                    "hybrid_score",
                    0.0
                ),
                3
            )

            rerank_score = round(
                chunk.get(
                    "rerank_score",
                    0.0
                ),
                3
            )

            evidence_markdown += f"""
### Evidence {number}

**Source:** {chunk['source_file']}
**Document Type:** {chunk['document_type']}
**Page:** {chunk['page_start']}
**Chunk ID:** `{chunk['chunk_id']}`
**Hybrid Score:** {hybrid_score}
**Reranker Score:** {rerank_score}

> {chunk['text']}

---
"""

    return (
        history,
        "",
        details,
        evidence_markdown
    )


# =========================================================
# CLEAR CHAT
# =========================================================

def clear_chat_ui():

    return (
        [],
        "",
        "No answer details yet.",
        "No evidence retrieved yet."
    )


# =========================================================
# EVIDENCE EXPLORER FUNCTION
# =========================================================

def evidence_search_ui(
    query,
    top_k,
    document_filter
):

    if not query or not query.strip():

        return (
            "Enter a search query first.",
            pd.DataFrame()
        )

    try:

        route = route_query(
            query
        )

        evidence = retrieve_for_route(
            query=query,
            route=route,
            top_k=int(top_k),
            document_type_filter=
                document_filter
        )

        if not evidence:

            return (
                "No relevant evidence found.",
                pd.DataFrame()
            )

        rows = []

        markdown = ""

        for rank, chunk in enumerate(
            evidence,
            1
        ):

            hybrid_score = round(
                chunk.get(
                    "hybrid_score",
                    0.0
                ),
                4
            )

            rerank_score = round(
                chunk.get(
                    "rerank_score",
                    0.0
                ),
                4
            )

            rows.append({
                "Rank": rank,
                "Source":
                    chunk[
                        "source_file"
                    ],
                "Type":
                    chunk[
                        "document_type"
                    ],
                "Page":
                    chunk[
                        "page_start"
                    ],
                "Hybrid Score":
                    hybrid_score,
                "Reranker Score":
                    rerank_score,
                "OCR Used":
                    chunk[
                        "ocr_used"
                    ],
                "Chunk ID":
                    chunk[
                        "chunk_id"
                    ]
            })

            markdown += f"""
### #{rank} — {chunk['source_file']}

**Page:** {chunk['page_start']}
**Document Type:** {chunk['document_type']}
**Hybrid Score:** {hybrid_score}
**Reranker Score:** {rerank_score}

> {chunk['text']}

---
"""

        return (
            markdown,
            pd.DataFrame(
                rows
            )
        )

    except Exception as error:

        traceback.print_exc()

        return (
            f"### Evidence Search Error\n\n"
            f"**{type(error).__name__}:** "
            f"{str(error)}",
            pd.DataFrame()
        )


# =========================================================
# EVALUATION HELPERS
# =========================================================

def create_test_set_from_dataframe(
    dataframe
):

    if dataframe is None:
        return []

    if not isinstance(
        dataframe,
        pd.DataFrame
    ):

        dataframe = pd.DataFrame(
            dataframe
        )

    test_set = []

    for _, row in dataframe.iterrows():

        question = str(
            row.get(
                "Question",
                ""
            )
        ).strip()

        if not question:
            continue

        expected_answer = str(
            row.get(
                "Expected Answer",
                ""
            )
        ).strip()

        expected_source = str(
            row.get(
                "Expected Source",
                ""
            )
        ).strip()

        expected_page_value = row.get(
            "Expected Page",
            None
        )

        expected_page = None

        try:

            if (
                expected_page_value
                is not None
                and
                str(
                    expected_page_value
                ).strip()
                not in [
                    "",
                    "nan",
                    "None"
                ]
            ):

                expected_page = int(
                    float(
                        expected_page_value
                    )
                )

        except:

            expected_page = None

        test_set.append({
            "question":
                question,
            "expected_answer":
                expected_answer,
            "expected_source":
                expected_source,
            "expected_page":
                expected_page
        })

    return test_set


def evaluation_ui(
    evaluation_table
):

    try:

        test_set = (
            create_test_set_from_dataframe(
                evaluation_table
            )
        )

        if not test_set:

            return (
                "Please enter at least one "
                "ground-truth test question.",
                pd.DataFrame(),
                None
            )

        results_df, metrics = (
            run_evaluation(
                test_set,
                top_k=
                    DEFAULT_TOP_K
            )
        )

        summary = f"""
## Evaluation Results

### {metrics['questions_tested']} Questions Tested

| Metric | Result |
|---|---:|
| Answer Accuracy | {metrics['answer_accuracy'] * 100:.1f}% |
| Citation Accuracy | {metrics['citation_accuracy'] * 100:.1f}% |
| Recall@{DEFAULT_TOP_K} | {metrics['recall_at_k'] * 100:.1f}% |
| Precision@{DEFAULT_TOP_K} | {metrics['precision_at_k'] * 100:.1f}% |
| Mean Reciprocal Rank | {metrics['mrr']:.3f} |
| Avg Retrieval Time | {metrics['avg_retrieval_time']:.3f} sec |
| Avg Generation Time | {metrics['avg_generation_time']:.3f} sec |
| Avg Total Response Time | {metrics['avg_total_time']:.3f} sec |
"""

        export_path = (
            os.path.join(
                tempfile.gettempdir(),
                "documind_evaluation_results.csv"
            )
        )

        results_df.to_csv(
            export_path,
            index=False
        )

        return (
            summary,
            results_df,
            export_path
        )

    except Exception as error:

        traceback.print_exc()

        return (
            f"### Evaluation Error\n\n"
            f"**{type(error).__name__}:** "
            f"{str(error)}",
            pd.DataFrame(),
            None
        )


# =========================================================
# DEFAULT EVALUATION TABLE
# =========================================================

default_test_data = pd.DataFrame({
    "Question": [
        "",
        "",
        "",
        "",
        ""
    ],

    "Expected Answer": [
        "",
        "",
        "",
        "",
        ""
    ],

    "Expected Source": [
        "",
        "",
        "",
        "",
        ""
    ],

    "Expected Page": [
        "",
        "",
        "",
        "",
        ""
    ]
})


# =========================================================
# BUILD FINAL APP
# =========================================================

with gr.Blocks(
    title="DocuMind AI",
    css=custom_css
) as demo:


    # =====================================================
    # HEADER
    # =====================================================

    with gr.Row():

        with gr.Column(
            scale=4
        ):

            gr.Markdown(
                "# ◆ DocuMind AI",
                elem_id=
                    "brand-title"
            )

            gr.Markdown(
                """
                Pharmaceutical Document Intelligence
                & Evidence Assistant
                """,
                elem_id=
                    "brand-subtitle"
            )

        with gr.Column(
            scale=1
        ):

            gr.Markdown(
                "● SYSTEM READY",
                elem_id=
                    "system-ready"
            )


    # =====================================================
    # DASHBOARD METRICS
    # =====================================================

    with gr.Row():

        with gr.Column(
            elem_classes=
                "metric-card"
        ):

            gr.Markdown(
                "<div class='metric-label'>Documents</div>"
            )

            files_metric = gr.Markdown(
                "<div class='metric-value'>0</div>"
            )

        with gr.Column(
            elem_classes=
                "metric-card"
        ):

            gr.Markdown(
                "<div class='metric-label'>Pages</div>"
            )

            pages_metric = gr.Markdown(
                "<div class='metric-value'>0</div>"
            )

        with gr.Column(
            elem_classes=
                "metric-card"
        ):

            gr.Markdown(
                "<div class='metric-label'>Chunks</div>"
            )

            chunks_metric = gr.Markdown(
                "<div class='metric-value'>0</div>"
            )

        with gr.Column(
            elem_classes=
                "metric-card"
        ):

            gr.Markdown(
                "<div class='metric-label'>OCR Pages</div>"
            )

            ocr_metric = gr.Markdown(
                "<div class='metric-value'>0</div>"
            )


    # =====================================================
    # TAB 1 — DOCUMENT CHAT
    # =====================================================

    with gr.Tab(
        "Document Chat"
    ):

        with gr.Row():

            with gr.Column(
                scale=1,
                elem_classes="card"
            ):

                gr.Markdown(
                    "## Document Workspace"
                )

                gr.Markdown(
                    """
                    Upload pharmaceutical PDFs.
                    Digital pages are extracted directly,
                    while scanned pages use OCR.
                    """
                )

                pdf_files = gr.File(
                    label=
                        "Upload PDF Documents",
                    file_types=[
                        ".pdf"
                    ],
                    file_count=
                        "multiple",
                    type=
                        "filepath"
                )

                document_type_override = (
                    gr.Dropdown(
                        choices=[
                            "Automatic Detection",
                            "General Document",
                            "Certificate of Quality",
                            "Packaging Specification",
                            "BSE/TSE Declaration",
                            "Cover Letter",
                            "Compliance Document",
                            "Technical Report"
                        ],
                        value=
                            "Automatic Detection",
                        label=
                            "Document Type"
                    )
                )

                process_button = (
                    gr.Button(
                        "Process & Index Documents",
                        variant=
                            "primary",
                        elem_classes=
                            "primary-button"
                    )
                )

                processing_status = (
                    gr.Markdown(
                        """
                        No documents have been
                        processed yet.
                        """,
                        elem_id=
                            "processing-status"
                    )
                )

                with gr.Accordion(
                    "Document Processing Report",
                    open=False
                ):

                    document_report = (
                        gr.Markdown(
                            "No report available."
                        )
                    )

                gr.Markdown(
                    "## Retrieval Controls"
                )

                document_filter = (
                    gr.Dropdown(
                        choices=[
                            "All Documents",
                            "General Document",
                            "Certificate of Quality",
                            "Packaging Specification",
                            "BSE/TSE Declaration",
                            "Cover Letter",
                            "Compliance Document",
                            "Technical Report"
                        ],
                        value=
                            "All Documents",
                        label=
                            "Search Document Type"
                    )
                )

                top_k_slider = (
                    gr.Slider(
                        minimum=1,
                        maximum=8,
                        value=
                            DEFAULT_TOP_K,
                        step=1,
                        label=
                            "Evidence Chunks to Use"
                    )
                )

                auto_route_checkbox = (
                    gr.Checkbox(
                        value=True,
                        label=
                            "Enable Smart Query Routing"
                    )
                )


            # =============================================
            # CHAT PANEL
            # =============================================

            with gr.Column(
                scale=2,
                elem_classes="card"
            ):

                gr.Markdown(
                    "## AI Document Chat"
                )

                chatbot = gr.Chatbot(
                    label=
                        "Conversation",
                    height=500
                )

                answer_details = (
                    gr.Markdown(
                        "No answer details yet.",
                        elem_id=
                            "answer-details"
                    )
                )

                with gr.Row():

                    user_message = (
                        gr.Textbox(
                            placeholder=(
                                "Ask about storage "
                                "conditions, specifications, "
                                "compliance evidence..."
                            ),
                            label=
                                "Your Question",
                            lines=2,
                            scale=5
                        )
                    )

                    send_button = (
                        gr.Button(
                            "Send →",
                            variant=
                                "primary",
                            elem_classes=
                                "primary-button",
                            scale=1
                        )
                    )

                clear_button = (
                    gr.Button(
                        "Clear Conversation",
                        elem_classes=
                            "secondary-button"
                    )
                )

                with gr.Accordion(
                    "Retrieved Evidence",
                    open=False
                ):

                    chat_evidence = (
                        gr.Markdown(
                            "No evidence retrieved yet.",
                            elem_id=
                                "evidence-output"
                        )
                    )


    # =====================================================
    # TAB 2 — EVIDENCE EXPLORER
    # =====================================================

    with gr.Tab(
        "Evidence Explorer"
    ):

        gr.Markdown(
            """
            ## Evidence Explorer

            Inspect retrieval independently of the
            language model. This helps explain why
            an answer was generated.
            """
        )

        with gr.Row():

            evidence_query = (
                gr.Textbox(
                    label=
                        "Search Query",
                    placeholder=(
                        "Example: recommended "
                        "storage temperature"
                    ),
                    scale=4
                )
            )

            evidence_top_k = (
                gr.Slider(
                    minimum=1,
                    maximum=8,
                    value=4,
                    step=1,
                    label=
                        "Top K",
                    scale=1
                )
            )

        evidence_filter = (
            gr.Dropdown(
                choices=[
                    "All Documents",
                    "General Document",
                    "Certificate of Quality",
                    "Packaging Specification",
                    "BSE/TSE Declaration",
                    "Cover Letter",
                    "Compliance Document",
                    "Technical Report"
                ],
                value=
                    "All Documents",
                label=
                    "Document Type Filter"
            )
        )

        evidence_button = (
            gr.Button(
                "Search Evidence",
                variant="primary",
                elem_classes=
                    "primary-button"
            )
        )

        evidence_text = (
            gr.Markdown(
                "Run a search to inspect retrieved evidence."
            )
        )

        evidence_table = (
            gr.Dataframe(
                headers=[
                    "Rank",
                    "Source",
                    "Type",
                    "Page",
                    "Hybrid Score",
                    "Reranker Score",
                    "OCR Used",
                    "Chunk ID"
                ],
                interactive=False,
                label=
                    "Retrieval Results"
            )
        )


    # =====================================================
    # TAB 3 — EVALUATION LAB
    # =====================================================

    with gr.Tab(
        "Evaluation Lab"
    ):

        gr.Markdown(
            """
            ## Evaluation Lab

            Test DocuMind AI against manually verified
            ground truth and calculate retrieval,
            answer, citation and latency metrics.
            """
        )

        gr.Markdown(
            """
            Enter verified test questions below.

            **Expected Answer** should contain the
            key information that must appear.

            **Expected Source** should exactly match
            the uploaded PDF file name.

            **Expected Page** should contain the
            correct evidence.
            """
        )

        evaluation_input = (
            gr.Dataframe(
                value=
                    default_test_data,
                headers=[
                    "Question",
                    "Expected Answer",
                    "Expected Source",
                    "Expected Page"
                ],
                datatype=[
                    "str",
                    "str",
                    "str",
                    "str"
                ],
                row_count=(
                    5,
                    "dynamic"
                ),
                col_count=(
                    4,
                    "fixed"
                ),
                interactive=True,
                label=
                    "Ground Truth Test Set"
            )
        )

        run_evaluation_button = (
            gr.Button(
                "Run Full Evaluation",
                variant="primary",
                elem_classes=
                    "primary-button"
            )
        )

        evaluation_summary = (
            gr.Markdown(
                "No evaluation has been run yet.",
                elem_id=
                    "evaluation-summary"
            )
        )

        evaluation_results = (
            gr.Dataframe(
                interactive=False,
                label=
                    "Detailed Evaluation Results"
            )
        )

        evaluation_download = (
            gr.File(
                label=
                    "Download Evaluation Results",
                interactive=False
            )
        )


    # =====================================================
    # FOOTER
    # =====================================================

    gr.Markdown(
        """
        DocuMind AI • Open-source RAG prototype
        PyMuPDF • Tesseract OCR • MiniLM • BM25 •
        CrossEncoder • FAISS • FLAN-T5 • Gradio
        """,
        elem_id=
            "footer"
    )


    # =====================================================
    # CONNECT DOCUMENT PROCESSING
    # =====================================================

    process_button.click(
        fn=
            process_documents_ui,

        inputs=[
            pdf_files,
            document_type_override
        ],

        outputs=[
            processing_status,
            files_metric,
            pages_metric,
            chunks_metric,
            ocr_metric,
            document_report
        ]
    )


    # =====================================================
    # CONNECT CHAT
    # =====================================================

    send_button.click(
        fn=
            chat_ui,

        inputs=[
            user_message,
            chatbot,
            top_k_slider,
            document_filter,
            auto_route_checkbox
        ],

        outputs=[
            chatbot,
            user_message,
            answer_details,
            chat_evidence
        ]
    )

    user_message.submit(
        fn=
            chat_ui,

        inputs=[
            user_message,
            chatbot,
            top_k_slider,
            document_filter,
            auto_route_checkbox
        ],

        outputs=[
            chatbot,
            user_message,
            answer_details,
            chat_evidence
        ]
    )

    clear_button.click(
        fn=
            clear_chat_ui,

        outputs=[
            chatbot,
            user_message,
            answer_details,
            chat_evidence
        ]
    )


    # =====================================================
    # CONNECT EVIDENCE EXPLORER
    # =====================================================

    evidence_button.click(
        fn=
            evidence_search_ui,

        inputs=[
            evidence_query,
            evidence_top_k,
            evidence_filter
        ],

        outputs=[
            evidence_text,
            evidence_table
        ]
    )


    # =====================================================
    # CONNECT EVALUATION LAB
    # =====================================================

    run_evaluation_button.click(
        fn=
            evaluation_ui,

        inputs=[
            evaluation_input
        ],

        outputs=[
            evaluation_summary,
            evaluation_results,
            evaluation_download
        ]
    )


print(
    "Final DocuMind AI interface "
    "built successfully."
)

Final DocuMind AI interface built successfully.


In [ ]:
# Step 22: Launch DocuMind AI

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b140db5d0658a70913.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
